# 02 — Représentation et retrieval de schéma

Ce notebook examine les schémas extraits des bases SQLite et les documents indexés pour le RAG de schéma. La méthode retenue est évaluée dans le notebook 04.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCHEMAS_DIR = ROOT / 'data' / 'schemas'
schemas = []
for path in sorted(SCHEMAS_DIR.glob('*.json')):
    with path.open(encoding='utf-8') as stream:
        schemas.append(json.load(stream))
print(f'{len(schemas)} schémas chargés')
schemas[0].keys()

In [ ]:
def schema_stats(schema):
    tables = schema.get('tables', [])
    return {
        'db_id': schema.get('db_id', 'inconnu'),
        'tables': len(tables),
        'columns': sum(len(table.get('columns', [])) for table in tables),
        'foreign_keys': sum(len(table.get('foreign_keys', [])) for table in tables),
    }
stats = pd.DataFrame([schema_stats(schema) for schema in schemas]).sort_values('db_id')
stats

In [ ]:
DB_ID = stats.iloc[0]['db_id']
schema = next(item for item in schemas if item.get('db_id') == DB_ID)
rows = []
for table in schema.get('tables', []):
    for column in table.get('columns', []):
        rows.append({'table': table.get('name'), 'column': column.get('name'), 'type': column.get('type'), 'primary_key': column.get('primary_key', False)})
print(f'Exemple : {DB_ID}')
pd.DataFrame(rows).head(30)

In [ ]:
INDEX_DIR = ROOT / 'data' / 'index'
required = ['table_docs.json', 'table_bm25.pkl', 'table_embeddings.npy', 'column_docs.json', 'column_bm25.pkl', 'column_embeddings.npy']
pd.DataFrame({'fichier': required, 'présent': [(INDEX_DIR / name).exists() for name in required]})

## Reconstruction de l'index

Après toute modification des schémas, relancez depuis la racine du dépôt :

```powershell
python src/retriever_schema.py build --schemas_dir data/schemas --output_dir data/index
```

Les documents d'index intègrent tables, colonnes et relations de clés étrangères.